# Ch15. Foundation Models
**Forecasting: Principles & Practice (Python Edition)**  
Lab Notebook · [github.com/bcseong2/fpppy-labs](https://github.com/bcseong2/fpppy-labs)

In [ ]:
%pip install statsforecast neuralforecast hierarchicalforecast mlforecast utilsforecast nixtla


## [Slide 11] 15.2 NHITS Transfer Learning Example

In [ ]:
from datasetsforecast.m4 import M4
from neuralforecast.utils import AirPassengersDF

# Load M4 monthly data (source domain)
Y_df = M4.load(directory="data", group="Monthly")[0].assign(
    ds=lambda df: df.groupby("unique_id")["ds"].transform(
        lambda x: pd.date_range(
            start="1970-01-01", periods=len(x), freq="MS"
        )
    )
)
horizon = 12
stacks = 3
models = [NHITS(
    input_size=5 * horizon, h=horizon,
    max_steps=2_000,
    stack_types=stacks * ["identity"],
    n_blocks=stacks * [1],
    mlp_units=[[256, 256] for _ in range(stacks)],
    n_pool_kernel_size=stacks * [1],
    batch_size=32, scaler_type="standard",
    n_freq_downsample=[12, 4, 1],
)]
nf = NeuralForecast(models=models, freq="MS")
nf.fit(df=Y_df)
# Zero-shot on unseen Air Passengers (target domain)
transfer_preds = nf.predict(df=AirPassengersDF.copy())

## [Slide 19] 15.3 TimeGPT: 기본 골격 코드

*API 키가 필요합니다(`NIXTLA_API_KEY` 환경변수). 키가 없으면 이 칸은 건너뛰세요.*

In [ ]:
import os
import pandas as pd
from nixtla import NixtlaClient

# 1. 클라이언트: 학습(fit) 단계 없음, 키는 환경변수로
client = NixtlaClient(api_key=os.environ["NIXTLA_API_KEY"])
client.validate_api_key()
# 2. 자료: long format (unique_id, ds, y)
df = pd.read_csv("data/electricity_short.csv", parse_dates=["ds"])
# 3. zero-shot 예측 (열: TimeGPT, TimeGPT-lo-80, ...)
fc = client.forecast(df=df, h=24, freq="h", level=[80, 90])
# 4. 교차검증 (statsforecast와 같은 인자)
cv = client.cross_validation(df=df, h=24, n_windows=3)
# 미세조정: finetune_steps=15 / 외생변수: X_df= 추가
fc.head()

## [Slide 20] 15.3 Lag-Llama: 기본 골격 코드

*Needs `pip install git+https://github.com/time-series-foundation-models/lag-llama` (gluonts). Downloads the checkpoint from Hugging Face Hub.*

In [ ]:
# df: long DataFrame (unique_id, ds, y); y: 1-D array of one series
# pip install git+https://github.com/\
#   time-series-foundation-models/lag-llama
import torch
from huggingface_hub import hf_hub_download
from gluonts.dataset.pandas import PandasDataset
from gluonts.evaluation import make_evaluation_predictions
from lag_llama.gluon.estimator import LagLlamaEstimator

# 1. 가중치: Hugging Face Hub에서 체크포인트 내려받기
ckpt_path = hf_hub_download(
    "time-series-foundation-models/Lag-Llama", "lag-llama.ckpt")
kw = torch.load(ckpt_path, map_location="cpu")[
    "hyper_parameters"]["model_kwargs"]   # 구조 설정(층 수 등)
# 2. 자료: gluonts 형식 (long df: unique_id, ds, y)
ds = PandasDataset.from_long_dataframe(
    df, target="y", item_id="unique_id", timestamp="ds")
# 3. 추정기 -> 예측기 (학습 없음, ckpt_path로 가중치만 불러옴)
est = LagLlamaEstimator(
    ckpt_path=ckpt_path, prediction_length=12, context_length=32,
    input_size=kw["input_size"], n_layer=kw["n_layer"],
    n_head=kw["n_head"], n_embd_per_head=kw["n_embd_per_head"],
    scaling=kw["scaling"], time_feat=kw["time_feat"])
predictor = est.create_predictor(
    est.create_transformation(), est.create_lightning_module())
# 4. Student t 분포에서 표본 100개 -> 중앙값·분위수
fc_it, _ = make_evaluation_predictions(
    ds, predictor, num_samples=100)
fc = next(fc_it); fc.median, fc.quantile(0.9)

## [Slide 21] 15.3 TimesFM: 기본 골격 코드

*Needs `pip install "timesfm[torch]==1.3.0"` (Python ≥ 3.11). Downloads the 200M checkpoint (large). Skip if not installed.*

In [ ]:
# df: long DataFrame (unique_id, ds, y); y: 1-D array of one series
# pip install "timesfm[torch]==1.3.0"   (Python >= 3.11)
import timesfm

# 1. 모형: 가중치는 Hugging Face Hub에서 (200M, 학습 없음)
tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend="cpu", per_core_batch_size=32,
        horizon_len=12),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id=
        "google/timesfm-1.0-200m-pytorch"))
# 2. 배열 입력: 시계열 목록 + 빈도 코드
#    (0 = 일별 이하, 1 = 주·월별, 2 = 분기·연별)
point, quantiles = tfm.forecast([y], freq=[1])
# 3. DataFrame 입력 (unique_id, ds, y) -> 예측 DataFrame
fc = tfm.forecast_on_df(inputs=df, freq="M", value_name="y",
                        num_jobs=-1)
# 점 예측만 쓸 것: quantiles는 보정되지 않은 실험 기능

## [Slide 22] 15.3 Moirai: 기본 골격 코드

*Needs `pip install uni2ts`. Downloads the small checkpoint from Hugging Face Hub.*

In [ ]:
# df: long DataFrame (unique_id, ds, y); y: 1-D array of one series
# pip install uni2ts
from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

# 1. 자료: gluonts 형식, 마지막 12개를 시험 구간으로
ds = PandasDataset.from_long_dataframe(
    df, target="y", item_id="unique_id", timestamp="ds")
train, test_template = split(ds, offset=-12)
test = test_template.generate_instances(
    prediction_length=12, windows=1, distance=12)
# 2. 모형: 가중치는 Hugging Face Hub (small / base / large)
model = MoiraiForecast(
    module=MoiraiModule.from_pretrained(
        "Salesforce/moirai-1.1-R-small"),
    prediction_length=12, context_length=120,
    patch_size="auto", num_samples=100, target_dim=1,
    feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0)
# 3. 예측기 -> 혼합분포에서 표본 100개 -> 중앙값·분위수
predictor = model.create_predictor(batch_size=32)
fc = next(iter(predictor.predict(test.input)))
fc.median, fc.quantile(0.9)

## [Slide 23] 15.3 Chronos: 기본 골격 코드

*Needs `pip install chronos-forecasting`. Verified with `amazon/chronos-t5-tiny` on AirPassengers (chronos-forecasting 2.3.2: first argument is positional `inputs`, not `context=`).*

In [ ]:
# df: long DataFrame (unique_id, ds, y); y: 1-D array of one series
# pip install chronos-forecasting
import torch, numpy as np
from chronos import ChronosPipeline

# 1. 모형: 가중치는 Hugging Face Hub (t5-tiny ... t5-large)
pipe = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small", device_map="cpu",
    torch_dtype=torch.bfloat16)
# 2. 과거값 텐서 -> 다음 토큰 분포에서 표본 경로 20개
fc = pipe.predict(torch.tensor(y), prediction_length=12,
                  num_samples=20)      # shape [1, 20, 12]
# 3. 표본의 분위수 = 점 예측(중앙값)·예측구간
lo, med, hi = np.quantile(fc[0].numpy(), [0.1, 0.5, 0.9],
                          axis=0)
# 분위수를 바로: pipe.predict_quantiles(torch.tensor(y),
#     prediction_length=12, quantile_levels=[0.1, 0.5, 0.9])

## [Slide 24] 15.4 Example: Electricity Price Forecasting

In [ ]:
df = pd.read_csv(
    "data/electricity_short.csv", parse_dates=["ds"]
)

## [Slide 25] 15.4 TimeGPT Zero-Shot Forecasting

In [ ]:
nixtla_client = NixtlaClient(api_key="YOUR_API_KEY")

# Zero-shot: no fine-tuning, direct application
preds_df = nixtla_client.forecast(
    df=df, h=24, level=[80, 90]
)

# Cross-validation for honest evaluation
cv_preds_df = nixtla_client.cross_validation(
    df=df, h=24, n_windows=3
)

# Evaluate
evaluation = evaluate(
    cv_preds_df, models=["TimeGPT"], metrics=[mae]
)

## [Slide 26] 15.4 TimeGPT Fine-Tuning

In [ ]:
cv_finetune_preds_df = nixtla_client.cross_validation(
    df=df,
    h=24,
    n_windows=3,
    finetune_steps=15,    # number of gradient steps
    finetune_loss="mae",  # target metric to optimise
)

## [Slide 27] 15.4 TimeGPT with Exogenous Variables

In [ ]:
future_ex_vars_df = pd.read_csv(
    "data/electricity_future_vars.csv",
    parse_dates=["ds"],
)

# Forecast with future exogenous variables
exog_preds_df = nixtla_client.forecast(
    df=df,
    X_df=future_ex_vars_df,   # future covariates
    h=24,
    level=[80],
)

## [Slide 28] 15.4 Moirai Zero-Shot Forecasting

In [ ]:
from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule
import numpy as np

full_df = pd.concat([df, future_ex_vars_df], axis=0)
full_df = full_df.set_index("ds")
ds = PandasDataset.from_long_dataframe(
    full_df, target="y", item_id="unique_id",
    feat_dynamic_real=full_df.columns
        .drop(["unique_id", "y"]).tolist(),
)
train, test_template = split(ds, offset=-24)
test_data = test_template.generate_instances(
    prediction_length=24, windows=1, distance=24,
)
model = MoiraiForecast(
    module=MoiraiModule.from_pretrained(
        "Salesforce/moirai-1.0-R-large"
    ),
    prediction_length=24, context_length=240,
    patch_size="auto", num_samples=50,
    target_dim=1,
    feat_dynamic_real_dim=ds.num_feat_dynamic_real,
    past_feat_dynamic_real_dim=ds.num_past_feat_dynamic_real,
)
predictor = model.create_predictor(batch_size=32)
forecasts = list(predictor.predict(test_data.input))

## [Slide 29] 15.4 Moirai Predictions and Combining Forecasts

In [ ]:
fc = forecasts[0]
moirai_preds_df = (
    pd.DataFrame(
        np.quantile(fc.samples, [0.5, 0.1, 0.9], axis=0).T
    )
    .set_axis(["Moirai", "Moirai-lo-80",
               "Moirai-hi-80"], axis="columns")
    .assign(ds=fc.index.to_timestamp())
)

# Combine TimeGPT and Moirai forecasts
fcst_df = preds_df.merge(exog_preds_df).merge(moirai_preds_df)

plot_series(
    df, fcst_df, max_insample_length=100,
    xlabel="Hour", ylabel="Price",
    title="Nord Pool electricity price",
    palette="black_and_3color", rm_legend=False,
    legend_loc="outside lower center",
)

## [Slide 30] 15.4 Chronos Usage

In [ ]:
from chronos import ChronosPipeline
import torch

# Load pre-trained model (T5-small variant)
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="cpu",
    torch_dtype=torch.bfloat16,
)

# y: numpy array or list of historical values
forecast = pipeline.predict(
    context=torch.tensor(y).unsqueeze(0),
    prediction_length=12,
    num_samples=20,    # number of sample paths
)
# forecast shape: [num_series, num_samples, prediction_length]
low, median, high = np.quantile(
    forecast[0].numpy(), [0.1, 0.5, 0.9], axis=0
)